# Fase 3 — preferências confirmadas entre sessões

Este notebook usa o mesmo serviço do menu 9, sem LLM. As preferências são comandos explícitos do usuário, não inferências do chat. O usuário de demonstração não é autenticação real.


In [ ]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.preferencias import PreferenciasUsuario


## Gravar e corrigir
A versão cresce apenas quando o valor muda. A resposta do agente não altera esta memória.


In [ ]:
pasta = tempfile.TemporaryDirectory()
arquivo = Path(pasta.name) / 'memoria.sqlite3'
ana, bruno = Contexto('demo', 'ana'), Contexto('demo', 'bruno')
with Banco(arquivo) as banco:
    banco.preparar_demo()
    preferencias = PreferenciasUsuario(banco)
    assert preferencias.salvar(ana, 'estilo', 'curto')['versao'] == 1
    assert preferencias.salvar(ana, 'estilo', 'detalhado')['versao'] == 2
    assert preferencias.listar(bruno) == []
    print('Ana:', preferencias.listar(ana), '| Bruno:', preferencias.listar(bruno))


## Reinício, isolamento e esquecimento
Reabrir o SQLite conserva a preferência atual. Apagar elimina também as versões antigas dessa chave.


In [ ]:
with Banco(arquivo) as banco:
    preferencias = PreferenciasUsuario(banco)
    assert preferencias.listar(ana)[0]['valor'] == 'detalhado'
    assert preferencias.listar(bruno) == []
    assert preferencias.apagar(ana, 'estilo')['estado'] == 'apagada'
    assert preferencias.listar(ana) == []
    total = banco.con.execute("SELECT COUNT(*) FROM historico_preferencias WHERE usuario_id=? AND chave=?", ('ana', 'estilo')).fetchone()[0]
    assert total == 0
    print('Persistência e exclusão verificadas; histórico remanescente:', total)
pasta.cleanup()


## Teste no terminal
Execute `python executar.py --usuario ana`, abra a opção 9 e use `/lembrar estilo=detalhado`, `/memorias`, `/nova` e `/memorias`. Reinicie e confira novamente. Depois use `/esquecer-memoria estilo`. Sem API, esses comandos locais funcionam; a influência na resposta do modelo exige teste real separado.
